# 02 — Modelado
Entrena los 3 modelos a nivel de **carrera genérica** y los guarda en `Models\` con `joblib`, para que el dashboard solo los cargue (sin reentrenar).

| Modelo | Datos de entrenamiento | Artefacto |
|---|---|---|
| 1. Recomendación de carreras | Reglas de negocio parametrizadas (no requiere entrenamiento) | `recomendador_config.joblib` |
| 2. Predicción de ingresos al 4° año | 252 combinaciones carrera genérica × tipo de institución | `modelo_ingresos.joblib` |
| 3. Segmentación de carreras | 165 carreras genéricas | `segmentacion.joblib` |

In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

estadisticas = pd.read_parquet("Data\\Clean\\estadisticas_carrera.parquet")
dataset_generica = pd.read_parquet("Data\\Clean\\dataset_carrera_generica.parquet")

os.makedirs("Models", exist_ok=True)
print("estadisticas:", estadisticas.shape, "| dataset_generica:", dataset_generica.shape)

## 1. Modelo 2 — Predicción de ingresos al 4° año

**Especificación:** el modelo predice el ingreso bruto mensual al 4° año de titulación a partir de tres variables que el usuario final puede elegir directamente: **área de interés**, **carrera genérica** y **tipo de institución donde ejercería** (universidad, IP o CFT).

**Datos:** las 252 combinaciones oficiales SIES de carrera genérica × tipo de institución (`estadisticas_carrera`), con el ingreso al 4° año como target. Se comparan Regresión Lineal (baseline), Random Forest y Gradient Boosting con validación cruzada de 5 folds (R² y MAE).

**Valor agregado del modelo sobre una consulta directa a la tabla:** para combinaciones que SIES no publica (por mínimo de casos del SII), el modelo extrapola desde los patrones de área y tipo de institución.

In [ ]:
target = "4° año"
features_ingreso = ["Área", "Área Carrera Genérica", "Tipo de institución"]

df_ing = estadisticas.copy()
for c in features_ingreso:
    df_ing[c] = df_ing[c].astype(str)
df_ing[target] = df_ing[target].astype(float)
df_ing = df_ing.dropna(subset=[target])
print("Filas de entrenamiento:", len(df_ing))

encoder_ing = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X = encoder_ing.fit_transform(df_ing[features_ingreso])
y = df_ing[target].values

cv = KFold(n_splits=5, shuffle=True, random_state=42)
candidatos = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

tabla_cv = []
for nombre, modelo in candidatos.items():
    r2 = cross_val_score(modelo, X, y, cv=cv, scoring="r2").mean()
    mae = -cross_val_score(modelo, X, y, cv=cv, scoring="neg_mean_absolute_error").mean()
    tabla_cv.append({"Modelo": nombre, "R2_CV": round(r2, 3), "MAE_CV": round(mae)})

tabla_cv = pd.DataFrame(tabla_cv).sort_values("R2_CV", ascending=False).reset_index(drop=True)
tabla_cv

In [ ]:
mejor_nombre = tabla_cv.loc[0, "Modelo"]
modelo_ingresos = candidatos[mejor_nombre]
modelo_ingresos.fit(X, y)

joblib.dump({
    "modelo": modelo_ingresos,
    "encoder": encoder_ing,
    "features": features_ingreso,
    "target": target,
    "nombre_algoritmo": mejor_nombre,
    "r2_cv": float(tabla_cv.loc[0, "R2_CV"]),
    "mae_cv": float(tabla_cv.loc[0, "MAE_CV"]),
    "n_train": len(df_ing),
}, "Models\\modelo_ingresos.joblib")

# Combinaciones disponibles para el simulador del dashboard
# (área -> carreras del área -> tipos donde se imparte, con el valor oficial SIES para contraste)
combos = df_ing[features_ingreso + [target]].rename(columns={target: "ingreso_4to_observado"})
combos.to_parquet("Data\\Clean\\combos_ingresos.parquet", index=False)

print(f"Guardado Models\\modelo_ingresos.joblib ({mejor_nombre})")

## 2. Modelo 3 — Segmentación de carreras genéricas

K-Means sobre las 165 carreras genéricas, con variables de retorno de inversión, empleabilidad, retención y brecha de duración. El k se elige por Silhouette Score (criterio principal de la rúbrica); las métricas complementarias (Davies-Bouldin, Calinski-Harabasz) se reportan en `03_Evaluation`.

In [ ]:
features_segmentacion = [
    "costo_total_carrera", "ingreso_4to_anio_valor",
    "Empleabilidad 1er año", "Retención 1er año", "brecha_duracion",
]

df_seg = dataset_generica.dropna(subset=features_segmentacion).copy()
scaler_seg = StandardScaler()
X_seg = scaler_seg.fit_transform(df_seg[features_segmentacion])

sil_scores = {}
for k in range(2, 8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_seg)
    sil_scores[k] = round(silhouette_score(X_seg, labels), 4)

print("Silhouette por k:", sil_scores)
k_optimo = max(sil_scores, key=sil_scores.get)
print("k óptimo:", k_optimo)

kmeans_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
df_seg["segmento"] = kmeans_final.fit_predict(X_seg)

joblib.dump({
    "kmeans": kmeans_final,
    "scaler": scaler_seg,
    "features": features_segmentacion,
    "k": int(k_optimo),
    "silhouette": sil_scores[k_optimo],
    "n_train": len(df_seg),
}, "Models\\segmentacion.joblib")

df_seg.groupby("segmento")[features_segmentacion].mean().round(2)

## 3. Modelo 1 — Recomendador de carreras

El recomendador es un **sistema de reglas de negocio con scoring**, no un modelo entrenado:
1. Filtra por área de interés del estudiante.
2. Factibilidad por puntaje PAES: corte referencial de la carrera ≤ puntaje del estudiante (las carreras sin corte publicado — impartidas solo en IP/CFT sin requisito PAES — son una vía alternativa).
3. Score final: empleabilidad 1er año (40%) + ingreso al 4° año (40%) + selectividad alcanzable (20%).

Se persisten los parámetros del score para que el dashboard los cargue igual que los otros modelos, dejando en `02` la única fuente de verdad de la configuración.

In [ ]:
config_recomendador = {
    "pesos": {"empleabilidad": 0.4, "ingreso": 0.4, "selectividad": 0.2},
    "col_corte": "puntaje_corte_paes",
    "descripcion": "Recomendador por reglas: factibilidad PAES + score empleabilidad/ingreso/selectividad",
}
joblib.dump(config_recomendador, "Models\\recomendador_config.joblib")
print("Guardado Models\\recomendador_config.joblib")
print("\nArtefactos en Models\\:", os.listdir("Models"))